# 00.0 — Data Quality Contract & Scientific Charter

## 1. Mục tiêu
Thiết lập bản Hiến chương Kiểm định Chất lượng Dữ liệu (**Data Quality Charter**) và phân định ranh giới trách nhiệm rành mạch:

$$\boxed{\text{Data Quality} \neq \text{EDA} \neq \text{Feature Engineering}}$$

Data Quality trả lời câu hỏi cốt lõi:
> **"Dữ liệu có hợp lệ, nhất quán và đủ tin cậy để đưa sang các bước tiếp theo hay không?"**

---

## 2. Nguyên tắc Phân định Ranh giới
- **“Tôi đang sửa vì dữ liệu SAI / HỎNG / KHUYẾT THỜI GIAN?”** $\longrightarrow$ **Data Quality**
- **“Tôi đang phân tích để hiểu QUY LUẬT / MỐI QUAN HỆ / REDUNDANCY?”** $\longrightarrow$ **EDA**
- **“Tôi đang biến quyết định thành INPUT CHO MÔ HÌNH?”** $\longrightarrow$ **Feature Engineering**

---

## 3. Quy tắc Vàng: Không Lạm quyền Modeling
1. Data Quality **KHÔNG** đưa ra các quyết định exploratory/modeling như:
   - $corr(X, Y) \text{ thấp} \Rightarrow drop(X)$
   - $VIF(X) > 10 \Rightarrow drop(X)$
   - $\text{feature importance thấp} \Rightarrow drop(X)$
2. Đặc biệt:
   $$\boxed{\text{low target correlation} \neq \text{bad data quality}}$$
3. Phân biệt Feature Redundancy:
   - *Trường hợp A (Duplicate sensor signal do lỗi ingestion/sinh dữ liệu)*: Xử lý ở Data Quality.
   - *Trường hợp B (Sensor hợp lệ nhưng tương quan cao với biến khác)*: Phân tích evidence tại **EDA**, thực thi drop/select tại **Feature Engineering**.

---

## 4. Danh mục Artifacts của Domain Data Quality
- `00.1` $\to$ `outputs/ingestion_audit_report.json`
- `00.2` $\to$ `outputs/missing_diagnosis_report.json`
- `00.3` $\to$ `outputs/canonical_split_report.json`
- `00.4` $\to$ `data/processed/clean_weather_data.csv` & `data_quality_report.json`


In [1]:
import sys
from pathlib import Path
import json

# Reconfigure encoding for safety
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config.constants import Config

print("=" * 70)
print("📜 00.0: DATA QUALITY CONTRACT & CHARTER")
print("=" * 70)

# Kiểm tra file dữ liệu thô bất biến
raw_csv_path = Path(Config.DATA_DIR) / f"hcmc_weather_data_daily_{Config.DEFAULT_START_DATE}_{Config.DEFAULT_END_DATE}.csv"
assert raw_csv_path.exists(), f"Raw CSV file does not exist: {raw_csv_path}"

print(f"✅ Raw Data File: {raw_csv_path.name}")
print(f"   • Immutable Raw Source Path: {raw_csv_path}")
print(f"   • Target Variable: 'Lượng mưa' (Daily Precipitation in mm)")
print(f"   • Expected Baseline Range: 2000-01-01 to 2026-04-30 (9,617 days, 36 features)")


📜 00.0: DATA QUALITY CONTRACT & CHARTER
✅ Raw Data File: hcmc_weather_data_daily_20000101_20260430.csv
   • Immutable Raw Source Path: D:\DS108_project\data\raw\hcmc_weather_data_daily_20000101_20260430.csv
   • Target Variable: 'Lượng mưa' (Daily Precipitation in mm)
   • Expected Baseline Range: 2000-01-01 to 2026-04-30 (9,617 days, 36 features)


In [2]:
# Xuất hợp đồng Data Quality Contract
OUTPUT_DIR = ROOT_DIR / "notebooks" / "data_quality" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contract = {
    "module": "Data Quality",
    "charter": "Data Integrity Gate (Validate, Diagnose, Repair Genuine Data Defects)",
    "boundary_rules": {
        "rule_1": "Data Quality only handles data validity, schema, duplicates, temporal continuity, and structural missingness.",
        "rule_2": "Data Quality NEVER drops features based on low correlation with target, VIF, or feature importance.",
        "rule_3": "Low target correlation != bad data quality.",
        "rule_4": "Feature redundancy analysis belongs to EDA; feature selection execution belongs to Feature Engineering."
    },
    "artifacts_registry": {
        "00.1": "ingestion_audit_report.json",
        "00.2": "missing_diagnosis_report.json",
        "00.3": "canonical_split_report.json",
        "00.4": "data_quality_report.json & clean_weather_data.csv"
    }
}

contract_path = OUTPUT_DIR / "data_quality_contract.json"
with open(contract_path, "w", encoding="utf-8") as f:
    json.dump(contract, f, indent=2, ensure_ascii=False)

print(f"💾 Data Quality Contract exported to: {contract_path}")
print("   • Status: APPROVED & FROZEN")


💾 Data Quality Contract exported to: D:\DS108_project\notebooks\data_quality\outputs\data_quality_contract.json
   • Status: APPROVED & FROZEN
